In [ ]:
import numpy as np
import torch
from predict import GaugeDataModel, Trainer, CNN_LSTM, get_sites_in_json
from nwis_downloader import create_site_info_df, cluster_sites, get_site_subsamples
from sklearn.metrics import r2_score
from matplotlib.pyplot import plt


In [ ]:
data_files = [{'path': 'site_dict.json',
               'conversion_factor': 0.0283168466,
               'data_key': 'discharge'}]
target_site = "07374000"
start_date = "2010-01-01"
end_date = "2025-01-01"
tz="UTC"
sequence_length = 90
forecast_horizon = 15
cutoff_date = np.datetime64('2020-01-01')
na_filter = 0.25
conversion_factor = 0.0283168466
epochs = 100
cluster_columns = ['log_drain_area', 'dec_lat_va', 'dec_long_va']
n_clusters = 50


In [ ]:
all_sites = get_sites_in_json(data_files['path'])
site_info = create_site_info_df(all_sites)
site_info['log_drain_area'] = np.log10(site_info['drain_area_va'])
clustered_sites = cluster_sites(site_info, cluster_columns, n_clusters)
subsamples = get_site_subsamples(clusterd_sites)


In [ ]:
for i, subsample in enumerate(subsamples):
    Gdm = GaugeDataModel(data_files,
                     target_site,
                     start_date,
                     end_date,
                     tz,
                     sequence_length,
                     forecast_horizon,
                     cutoff_date,
                     site_filter=subsample
                     )
    gdm.setup()
    model = CNN_LSTM(gdm.train_dataset.input_channels, sequence_length)
    optimizer = torch.optim.Adam(model.parameters(), lr=1.15e-6, weight_decay=0.5e-4)
    criterion = torch.nn.MSELoss()
    trainer = Trainer(model, gdm, gdm.scaler_y, criterion, optimizer)
    trainer.fit(epochs)
    weight_path = os.path.join(weight_dir, f"gp_subset_{i}.pt")
    torch.save(trainer.model.state_dict(), weight_path)
